# UrbanSound8K Classification — Colab Pro

**Hardware options (Runtime → Change runtime type):**
- `T4 GPU` — Free / Pro: baseline option, ~16 GB VRAM
- `A100 GPU` — Pro+: fastest, use if available
- `V100 GPU` — Pro: good middle ground
- `TPU v2-8` — Experimental only; PyTorch on TPU requires extra setup, **use GPU instead**

**Recommended for this project: T4 or A100 GPU**

This notebook persists all checkpoints to Google Drive.
Run it across multiple sessions — completed folds are skipped automatically.

In [1]:
# ── Cell 0: Clone repo from GitHub ──────────────────────────────
import os
from google.colab import userdata

# Pull token from Colab Secrets (never hardcode it)
token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "dun011@ucsd.edu"
!git config --global user.name "DanielNg520

REPO = 'DanielNg520/Urbansound8k-spectrogram-CNN'  # ← edit this

# Clone or pull latest
if os.path.exists('/content/urbansound'):
    # Already cloned — just pull latest changes
    %cd /content/urbansound
    !git pull https://{token}@github.com/{REPO}.git main
else:
    !git clone https://{token}@github.com/{REPO}.git /content/urbansound
    %cd /content/urbansound

# Add to Python path so core/ imports work
import sys
sys.path.insert(0, '/content/urbansound')

print('Repo ready.')

/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file
Cloning into '/content/urbansound'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 28 (delta 10), reused 24 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 41.62 KiB | 10.40 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/urbansound
Repo ready.


In [2]:
# ── Cell 1: Check hardware ──────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {gpu_mem:.1f} GB')
    DEVICE = torch.device('cuda')
else:
    print('No GPU! Go to Runtime → Change runtime type → GPU')
    DEVICE = torch.device('cpu')

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

GPU: Tesla T4
VRAM: 15.6 GB
Device: cuda
PyTorch: 2.10.0+cu128


In [3]:
# ── Cell 2: Mount Drive & set paths ────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT     = '/content/drive/MyDrive/ECE176_project'
DATASET_ROOT   = os.path.join(DRIVE_ROOT, 'UrbanSound8K')   # ← this is the fix
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
RESULTS_DIR    = os.path.join(DRIVE_ROOT, 'results')
CACHE_DIR      = os.path.join(DRIVE_ROOT, 'cache')

for d in [CHECKPOINT_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'Dataset: {DATASET_ROOT}')
print(f'Dataset exists: {os.path.exists(DATASET_ROOT)}')
print(f'CSV exists: {os.path.exists(os.path.join(DATASET_ROOT, "metadata", "UrbanSound8K.csv"))}')

Mounted at /content/drive
Drive mounted.
Dataset: /content/drive/MyDrive/ECE176_project/UrbanSound8K
Dataset exists: True
CSV exists: True


In [4]:
# ── Cell 3: Dataset setup ────────────────────────────────────────
import os
import shutil
import kagglehub
from google.colab import userdata
import json

dst = '/content/drive/MyDrive/ECE176_project/UrbanSound8K'
csv_path = os.path.join(dst, 'metadata', 'UrbanSound8K.csv')

if os.path.exists(csv_path):
    print('Dataset already in Drive with correct structure. Skipping.')
else:
    # Set Kaggle credentials
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({
            "username": userdata.get('KAGGLE_USERNAME'),
            "key":      userdata.get('KAGGLE_KEY')
        }, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

    # Download
    print('Downloading UrbanSound8K...')
    src = kagglehub.dataset_download('chrisfilo/urbansound8k')
    print(f'Downloaded to: {src}')

    # Delete any previous bad copy
    if os.path.exists(dst):
        print('Removing previous copy...')
        shutil.rmtree(dst)

    # Rebuild correct structure
    os.makedirs(os.path.join(dst, 'audio'), exist_ok=True)
    os.makedirs(os.path.join(dst, 'metadata'), exist_ok=True)

    for item in os.listdir(src):
        full = os.path.join(src, item)
        if item.startswith('fold'):
            print(f'  Copying {item}...')
            shutil.copytree(full, os.path.join(dst, 'audio', item))
        elif item.endswith('.csv'):
            print(f'  Copying {item}...')
            shutil.copy2(full, os.path.join(dst, 'metadata', 'UrbanSound8K.csv'))

    # Verify
    print(f'\nmetadata/UrbanSound8K.csv: {os.path.exists(csv_path)}')
    print(f'audio/ folders: {sorted(os.listdir(os.path.join(dst, "audio")))}')
    print('Done.')

Dataset already in Drive with correct structure. Skipping.


In [5]:
# ── Cell 4: Install dependencies ────────────────────────────────
!pip install -q librosa scikit-learn tqdm

In [6]:
# ── Cell 5: Core utilities (inline — no local files needed) ─────
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json

# ── Constants ──
CLASSES = ['air_conditioner','car_horn','children_playing','dog_bark',
           'drilling','engine_idling','gun_shot','jackhammer','siren','street_music']
SAMPLE_RATE = 22050
CLIP_DURATION = 4.0
N_MFCC = 40
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 2048
MEL_LENGTH = 128

# ── Audio loading ──
def load_audio(path, sr=SAMPLE_RATE, duration=CLIP_DURATION):
    try:
        y, _ = librosa.load(path, sr=sr, duration=duration, mono=True)
    except:
        return np.zeros(int(sr * duration), dtype=np.float32)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)

# ── Feature extraction ──
def extract_mfcc(y, sr=SAMPLE_RATE):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

def extract_mel(y, sr=SAMPLE_RATE, fixed_length=MEL_LENGTH):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS,
                                          n_fft=N_FFT, hop_length=HOP_LENGTH, fmax=8000)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    if log_mel.shape[1] < fixed_length:
        log_mel = np.pad(log_mel, ((0,0),(0, fixed_length - log_mel.shape[1])),
                         constant_values=log_mel.min())
    else:
        log_mel = log_mel[:, :fixed_length]
    lo, hi = log_mel.min(), log_mel.max()
    if hi - lo > 1e-6:
        log_mel = (log_mel - lo) / (hi - lo)
    return log_mel.astype(np.float32)

# ── Metadata ──
def load_metadata(root):
    return pd.read_csv(os.path.join(root, 'metadata', 'UrbanSound8K.csv'))

def audio_path(root, fold, fname):
    return os.path.join(root, 'audio', f'fold{fold}', fname)

def get_fold_splits(meta):
    for fold in sorted(meta['fold'].unique()):
        yield fold, meta[meta['fold'] != fold].copy(), meta[meta['fold'] == fold].copy()

print('Core utilities loaded.')

Core utilities loaded.


In [7]:
# ── Cell 6: Dataset + Models ─────────────────────────────────────

class MelDataset(Dataset):
    def __init__(self, df, root, augment=False):
        self.df = df.reset_index(drop=True)
        self.root = root
        self.augment = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = load_audio(audio_path(self.root, row['fold'], row['slice_file_name']))
        if self.augment:
            shift = int(np.random.uniform(-0.1, 0.1) * len(y))
            y = np.roll(y, shift)
            y += np.random.normal(0, 0.005, y.shape).astype(y.dtype)
        mel = torch.from_numpy(extract_mel(y)).unsqueeze(0)
        return mel, torch.tensor(int(row['classID']), dtype=torch.long)


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.MaxPool2d(2) if pool else nn.Identity()
        )
    def forward(self, x): return self.block(x)


class UrbanCNN(nn.Module):
    def __init__(self, n_classes=10, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1,   32),
            ConvBlock(32,  64),
            ConvBlock(64,  128),
            ConvBlock(128, 256),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*8*8, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_classes)
        )
    def forward(self, x): return self.head(self.features(x))


print('Dataset and model classes defined.')

Dataset and model classes defined.


In [8]:
# ── Cell 7: Config ───────────────────────────────────────────────
# Adjust these based on your GPU
# T4:   BATCH_SIZE=64,  NUM_WORKERS=2
# V100: BATCH_SIZE=128, NUM_WORKERS=4
# A100: BATCH_SIZE=256, NUM_WORKERS=4

BATCH_SIZE  = 64     # ← change based on GPU above
EPOCHS      = 60
LR          = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2      # ← change based on GPU above

PROGRESS_FILE = os.path.join(CHECKPOINT_DIR, 'progress.json')

print(f'Batch size: {BATCH_SIZE}')
print(f'Epochs: {EPOCHS}')
print(f'Progress file: {PROGRESS_FILE}')

Batch size: 64
Epochs: 60
Progress file: /content/drive/MyDrive/ECE176_project/checkpoints/progress.json


In [9]:
# ── Cell 8: Training helpers ─────────────────────────────────────
from sklearn.metrics import accuracy_score
import numpy as np

def train_epoch(model, loader, opt, criterion, scaler=None):
    model.train()
    total_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                out = model(x); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            out = model(x); loss = criterion(out, y)
            loss.backward(); opt.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        preds.extend(model(x.to(DEVICE)).argmax(1).cpu().tolist())
        labels.extend(y.tolist())
    return np.array(labels), np.array(preds)

def load_progress():
    """Load progress JSON. Falls back to .bak if primary is corrupted/empty."""
    def _try_load(path):
        if not os.path.exists(path):
            return None
        try:
            with open(path) as f:
                content = f.read().strip()
            if not content:
                return None
            return json.loads(content)
        except (json.JSONDecodeError, OSError):
            return None

    data = _try_load(PROGRESS_FILE)
    if data is not None:
        return data

    # Primary failed — try backup
    bak = PROGRESS_FILE + '.bak'
    data = _try_load(bak)
    if data is not None:
        print(f'[WARN] Primary progress corrupted, loaded from backup ({bak})')
        return data

    print('[WARN] No valid progress file found, starting fresh')
    return {'completed_folds': [], 'fold_results': []}

def save_progress(prog):
    """Atomically save progress JSON and rotate backup."""
    tmp = PROGRESS_FILE + '.tmp'
    bak = PROGRESS_FILE + '.bak'
    # Write to temp first
    with open(tmp, 'w') as f:
        json.dump(prog, f, indent=2)
    # Rotate: current → .bak, then tmp → current
    if os.path.exists(PROGRESS_FILE):
        os.replace(PROGRESS_FILE, bak)
    os.replace(tmp, PROGRESS_FILE)

def safe_torch_load(path, map_location=None):
    """Load a torch checkpoint safely, falling back to .bak if primary fails."""
    import numpy as np
    # Allow numpy scalars that get embedded by older torch.save calls
    try:
        torch.serialization.add_safe_globals([np.core.multiarray.scalar])
    except AttributeError:
        pass  # numpy version doesn't have this path

    def _load(p):
        return torch.load(p, map_location=map_location, weights_only=False)

    try:
        return _load(path)
    except Exception as e:
        bak = path + '.bak'
        if os.path.exists(bak):
            print(f'[WARN] Checkpoint {path} failed ({e}), trying backup...')
            return _load(bak)
        raise

def safe_torch_save(state, path):
    """Atomically save a torch checkpoint and keep a .bak copy."""
    tmp = path + '.tmp'
    bak = path + '.bak'
    torch.save(state, tmp)
    if os.path.exists(path):
        os.replace(path, bak)
    os.replace(tmp, path)

print('Training helpers defined.')


Training helpers defined.


In [10]:
# ── Cell 9: SVM Baseline ─────────────────────────────────────────
# Run this once — takes ~15-30 min on Colab CPU
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SVM_RESULTS_FILE = os.path.join(RESULTS_DIR, 'svm_results.json')

if os.path.exists(SVM_RESULTS_FILE):
    print('SVM results already exist. Loading from Drive...')
    with open(SVM_RESULTS_FILE) as f:
        svm_summary = json.load(f)
    print(f"SVM Mean Accuracy: {svm_summary['summary']['mean_accuracy']*100:.2f}%")
else:
    meta = load_metadata(DATASET_ROOT)

    # Pre-compute MFCC
    mfcc_cache = os.path.join(CACHE_DIR, 'mfcc.npz')
    if os.path.exists(mfcc_cache):
        data = np.load(mfcc_cache)
        X_all, y_all, folds_all = data['X'], data['y'], data['folds']
        print('Loaded MFCC cache.')
    else:
        X_all, y_all, folds_all = [], [], []
        for i, row in meta.iterrows():
            path = audio_path(DATASET_ROOT, row['fold'], row['slice_file_name'])
            audio = load_audio(path)
            X_all.append(extract_mfcc(audio))
            y_all.append(row['classID'])
            folds_all.append(row['fold'])
            if (i+1) % 1000 == 0: print(f'  MFCC {i+1}/{len(meta)}')
        X_all = np.array(X_all, dtype=np.float32)
        y_all = np.array(y_all, dtype=np.int64)
        folds_all = np.array(folds_all)
        np.savez(mfcc_cache, X=X_all, y=y_all, folds=folds_all)
        print('MFCC cache saved.')

    svm_fold_results = []
    for fold, train_df, test_df in get_fold_splits(meta):
        clf = Pipeline([('sc', StandardScaler()),
                         ('svm', SVC(kernel='rbf', C=10, gamma='scale', cache_size=1000))])
        tm = folds_all != fold
        te = folds_all == fold
        print(f'  SVM fold {fold}...')
        clf.fit(X_all[tm], y_all[tm])
        preds = clf.predict(X_all[te])
        acc = accuracy_score(y_all[te], preds)
        svm_fold_results.append({'fold': int(fold), 'accuracy': float(acc)})
        print(f'  Fold {fold}: {acc*100:.2f}%')

    accs = [r['accuracy'] for r in svm_fold_results]
    svm_summary = {
        'model': 'SVM+MFCC',
        'summary': {'mean_accuracy': float(np.mean(accs)), 'std_accuracy': float(np.std(accs))},
        'folds': svm_fold_results
    }
    with open(SVM_RESULTS_FILE, 'w') as f: json.dump(svm_summary, f, indent=2)
    print(f"\nSVM Mean: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%")

SVM results already exist. Loading from Drive...
SVM Mean Accuracy: 66.10%


In [12]:
# ── Cell 10: CNN Training (multi-session) ────────────────────────
# Safe to re-run: skips completed folds automatically

import os, json

meta = load_metadata(DATASET_ROOT)
prog = load_progress()
completed = set(prog['completed_folds'])
fold_results = prog['fold_results']

# Use non-deprecated GradScaler API
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

for test_fold, train_df, test_df in get_fold_splits(meta):
    if test_fold in completed:
        matching = [r for r in fold_results if r.get('fold') == test_fold]
        if matching:
            print(f'[SKIP] Fold {test_fold} — {matching[0]["accuracy"]*100:.2f}%')
        continue

    print(f'\n── Fold {test_fold} ── {len(train_df)} train / {len(test_df)} test')

    ckpt_best   = os.path.join(CHECKPOINT_DIR, f'fold{test_fold}_best.pt')
    ckpt_resume = os.path.join(CHECKPOINT_DIR, f'fold{test_fold}_resume.pt')

    train_ds = MelDataset(train_df, DATASET_ROOT, augment=True)
    test_ds  = MelDataset(test_df,  DATASET_ROOT, augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    model = UrbanCNN().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss()

    start_epoch = 1
    best_acc = 0.0

    if os.path.exists(ckpt_resume) or os.path.exists(ckpt_resume + '.bak'):
        try:
            state = safe_torch_load(ckpt_resume, map_location=DEVICE)
            model.load_state_dict(state['model'])
            opt.load_state_dict(state['optimizer'])
            sched.load_state_dict(state['scheduler'])
            start_epoch = state['epoch'] + 1
            best_acc = state['best_acc']
            print(f'Resumed from epoch {start_epoch}, best_acc={best_acc*100:.2f}%')
        except Exception as e:
            print(f'[WARN] Could not load resume checkpoint: {e}. Starting fold from scratch.')

    for epoch in range(start_epoch, EPOCHS + 1):
        loss, train_acc = train_epoch(model, train_loader, opt, crit, scaler)
        sched.step()

        if epoch % 5 == 0 or epoch == EPOCHS:
            # Atomically save resume checkpoint + .bak fallback
            safe_torch_save(
                {'epoch': epoch, 'model': model.state_dict(),
                 'optimizer': opt.state_dict(), 'scheduler': sched.state_dict(),
                 'best_acc': best_acc},
                ckpt_resume
            )

            labels, preds = eval_model(model, test_loader)
            val_acc = (labels == preds).mean()
            print(f'  E{epoch:02d}/{EPOCHS}  loss={loss:.4f}  '
                  f'train={train_acc*100:.1f}%  val={val_acc*100:.1f}%')

            if val_acc > best_acc:
                best_acc = val_acc
                # Atomically save best checkpoint + .bak fallback
                safe_torch_save(model.state_dict(), ckpt_best)

    # Final eval — load best checkpoint safely
    model.load_state_dict(safe_torch_load(ckpt_best, map_location=DEVICE))
    labels, preds = eval_model(model, test_loader)
    acc = float((labels == preds).mean())

    fold_results.append({'fold': int(test_fold), 'accuracy': acc})
    completed.add(int(test_fold))  # <--- FIXED: Explicitly cast to Python int
    prog['completed_folds'] = list(completed)
    prog['fold_results'] = fold_results
    save_progress(prog)  # atomic write + .bak rotation

    # Clean up resume checkpoints (keep .bak in case of re-run issues)
    for suffix in ('', '.bak', '.tmp'):
        p = ckpt_resume + suffix
        if os.path.exists(p): os.remove(p)
    torch.cuda.empty_cache()
    print(f'  Fold {test_fold} DONE — {acc*100:.2f}%')

print('\n── All folds complete ──')


'{\n  "completed_folds": [\n    '
Progress file corrupted, starting fresh

── Fold 1 ── 7859 train / 873 test


/tmp/ipython-input-1535404624.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [ ]:
# ── Cell 11: Final summary & comparison ─────────────────────────
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# CNN results
prog = load_progress()
cnn_accs = [r['accuracy'] for r in prog['fold_results']]
print('CNN Results:')
for r in sorted(prog['fold_results'], key=lambda x: x['fold']):
    print(f'  Fold {r["fold"]:2d}: {r["accuracy"]*100:.2f}%')
print(f'  Mean: {np.mean(cnn_accs)*100:.2f}% ± {np.std(cnn_accs)*100:.2f}%')

# SVM results
if os.path.exists(SVM_RESULTS_FILE):
    with open(SVM_RESULTS_FILE) as f: svm = json.load(f)
    svm_accs = [r['accuracy'] for r in svm['folds']]
    print(f"\nSVM Baseline: {np.mean(svm_accs)*100:.2f}% ± {np.std(svm_accs)*100:.2f}%")

# Save CNN results
cnn_summary = {
    'model': 'UrbanCNN',
    'summary': {'mean_accuracy': float(np.mean(cnn_accs)), 'std_accuracy': float(np.std(cnn_accs))},
    'folds': prog['fold_results']
}
with open(os.path.join(RESULTS_DIR, 'cnn_results.json'), 'w') as f:
    json.dump(cnn_summary, f, indent=2)
print('\nResults saved to Drive.')

In [ ]:
# ── Cell 12: Confusion matrix (run after all folds complete) ─────
# Re-runs inference on fold 10 for confusion matrix visualization
# You can pick any fold, or aggregate all

meta = load_metadata(DATASET_ROOT)
all_labels, all_preds = [], []

for test_fold, train_df, test_df in get_fold_splits(meta):
    ckpt = os.path.join(CHECKPOINT_DIR, f'fold{test_fold}_best.pt')
    if not os.path.exists(ckpt):
        print(f'Missing checkpoint for fold {test_fold}')
        continue

    test_ds = MelDataset(test_df, DATASET_ROOT, augment=False)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False,
                              num_workers=2, pin_memory=True)

    model = UrbanCNN().to(DEVICE)
    model.load_state_dict(safe_torch_load(ckpt, map_location=DEVICE))

    labels, preds = eval_model(model, test_loader)
    all_labels.extend(labels.tolist())
    all_preds.extend(preds.tolist())
    torch.cuda.empty_cache()

print(f'Overall accuracy: {accuracy_score(all_labels, all_preds)*100:.2f}%')

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — UrbanCNN (all folds)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('Saved to Drive.')

In [ ]:
#Extra Git Push
!git config --global user.email "dun011@ucsd.edu"
!git config --global user.name "DanielNg520"

!git add v3_colab_pro.ipynb
!git commit -m "fix DATASET_ROOT path"
!git push